# U-Mamba MTL (single-task) inference example

This notebook shows how to run the pretrained **umamba_mtl_single** model (5-fold ensemble) on a single
biparametric MRI (bpMRI) case:

- **Input:** T2w, ADC and HBV volumes (`.mha` or `.nii.gz`). The ADC and HBV volumes do **not** need to be
  resampled onto the T2w grid beforehand - the preprocessing (in `inference/inference_transforms.py`)
  resamples them to match.
- **Cropping:** no prostate segmentation is provided, so the pipeline defaults to a **center crop** of the image.
- **Model:** built from the checkpoint by `inference/build_umamba_mtl_single.py`
  (only the inference-relevant parts of the experiment's trainer and config).
- **Output:**
  - PCa detection probability map (`*_pca_prob.nii.gz`)
  - Zonal anatomy segmentation with PZ/TZ labels (`*_zonal_seg.nii.gz`)

Both predictions are stored on the original T2w grid and can optionally be saved as separate files.
The example uses the bundled PI-CAI sample case from `inference/sample_data/`, but the same cells work
on any bpMRI series.

**Requirements:**
- The repository environment (see the readme: `uv sync`), so the `umamba_mtl` package is importable.
- Run the notebook from its own directory (`inference/`) - the default when opening it in Jupyter - so the
  helper modules next to it and the relative paths below resolve correctly.
- The 5-fold ensemble weights in `weights/umamba_mtl_single/` (checked in the second cell).


In [8]:
from pathlib import Path

import torch

from build_umamba_mtl_single import build_umamba_mtl_single_ensemble, split_umamba_mtl_single_output
from inference_transforms import get_inference_transforms, get_post_transforms
from umamba_mtl.plotting import slice_comparison

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [9]:
# --- Settings -----------------------------------------------------------------
SAVE_PREDS = True                    # save the PCa probability map and zonal segmentation to OUTPUT_DIR
FOLDS = [0, 1, 2, 3, 4]              # cross-validation folds of the ensemble
WEIGHTS_DIR = Path("../weights/umamba_mtl_single")   # relative to this notebook (inference/)
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

missing = [name for name in (f"f{fold}.ckpt" for fold in FOLDS) if not (WEIGHTS_DIR / name).is_file()]
if missing:
    raise RuntimeError(
        f"Missing model weights: {missing}\n"
        f"Expected them in: {WEIGHTS_DIR.resolve()}\n\n"
        "Please download them first by running from the repository root:\n"
        "    uv run python utils/download_weights.py"
    )
print(f"Found all {len(FOLDS)} ensemble checkpoints in {WEIGHTS_DIR.resolve()}")

Found all 5 ensemble checkpoints in /data/work/michaesl/prostate/published_repos/umamba_mtl/weights/umamba_mtl_single


In [10]:
# Pre/post-processing transforms (defaults match the training configuration).
prep_spatial, prep_intensity = get_inference_transforms()
pca_post, zonal_post = get_post_transforms(prep_spatial, OUTPUT_DIR, save_preds=SAVE_PREDS)

# The 5-fold ensemble.
ensemble = build_umamba_mtl_single_ensemble(WEIGHTS_DIR, folds=FOLDS, device=DEVICE)
print(f"Loaded {len(ensemble)} models on {DEVICE}")

Loaded 5 models on cuda


## Input data

The pipeline expects **three separate volumes**: `t2w`, `adc` and `hbv` (`.mha` or `.nii.gz`). The ADC and HBV
volumes may be on a different grid than the T2w - they are resampled onto the T2w grid, which is then resampled
to the training spacing (0.5 x 0.5 x 3.0 mm).

The example below uses the raw PI-CAI MHA files bundled next to this notebook (in `sample_data/`;
the preprocessed `.nii.gz` files there work as well). To run inference on your own data, simply replace the
paths in `image_files` with your three files.

In [11]:
# Example case: the bundled PI-CAI sample data (raw MHA files).
CASE_ID = "10032_1000032"

image_files = {
    "t2w": "sample_data/mha/10032_1000032_t2w.mha",
    "adc": "sample_data/mha/10032_1000032_adc.mha",
    "hbv": "sample_data/mha/10032_1000032_hbv.mha",
}
# The preprocessed .nii.gz volumes (already resampled onto a common grid) work as well:
# image_files = {
#     "t2w": "sample_data/nifti/10032_1000032_0000.nii.gz",
#     "adc": "sample_data/nifti/10032_1000032_0001.nii.gz",
#     "hbv": "sample_data/nifti/10032_1000032_0002.nii.gz",
# }

data = prep_intensity(prep_spatial(image_files))
# Clean filename for the saved predictions.
data["image"].meta["filename_or_obj"] = f"{CASE_ID}.nii.gz"

print("Model input:", tuple(data["image"].shape))

Model input: (3, 256, 256, 20)


In [12]:
# Run the 5-fold ensemble and average the per-task probabilities.
x = data["image"][None].to(DEVICE)

pca_probs, zonal_probs = [], []
with torch.no_grad():
    for _, inferer in ensemble:
        logits = inferer(x)                                     # [1, 5, 256, 256, 20]
        pca, zonal = split_umamba_mtl_single_output(logits)
        pca_probs.append(pca[0].cpu())                          # [1, 256, 256, 20]
        zonal_probs.append(zonal[0].cpu())                      # [3, 256, 256, 20]

data["pca"] = torch.stack(pca_probs).mean(dim=0)      # ensemble-averaged PCa probability
data["zonal"] = torch.stack(zonal_probs).mean(dim=0)  # ensemble-averaged zonal probabilities
print("PCa probability:", tuple(data["pca"].shape),
      "| zonal probabilities:", tuple(data["zonal"].shape))

PCa probability: (1, 256, 256, 20) | zonal probabilities: (3, 256, 256, 20)


In [13]:
# Invert the predictions back to the original T2w grid (and save them if SAVE_PREDS).
pca_result = pca_post(data)
zonal_result = zonal_post(data)

pca_prob = pca_result["pca"]      # [1, X, Y, Z] PCa probabilities on the original T2w grid
zonal_seg = zonal_result["zonal"]  # [1, X, Y, Z] labels: 0=background, 1=PZ, 2=TZ

if SAVE_PREDS:
    print("Saved predictions to", OUTPUT_DIR)
    for path in sorted(OUTPUT_DIR.glob(f"{CASE_ID}*")):
        print(" -", path.name)
print("PCa probability range:", float(pca_prob.min()), "-", float(pca_prob.max()))

2026-11-24 16:26:31,848 INFO image_writer.py:197 - writing: output/10032_1000032_pca_prob.nii.gz
2026-11-24 16:26:32,322 INFO image_writer.py:197 - writing: output/10032_1000032_zonal_seg.nii.gz
Saved predictions to output
 - 10032_1000032_pca_prob.nii.gz
 - 10032_1000032_zonal_seg.nii.gz
PCa probability range: 0.0 - 0.818575382232666


## Visualization

Interactive overlay of the predictions on the original T2w using `slice_comparison` from
`umamba_mtl.plotting` - use the slider to scroll through the slices. Overlay values below
`alpha_threshold` (0.2 by default) are rendered transparent, so only sufficiently high
probabilities / confident labels cover the image.

In [ ]:
# PCa probability, predicted PZ and predicted TZ overlaid on the original T2w.
image = data["t2w"].detach().float()  # the original T2w on its original grid, (1, X, Y, Z)
labels = [
    pca_prob.detach().float(),       # PCa probability
    (zonal_seg.detach() == 1).float(),  # predicted PZ
    (zonal_seg.detach() == 2).float(),  # predicted TZ
]
titles = ["PCa probability", "PZ (predicted)", "TZ (predicted)"]

# Overlay values below alpha_threshold are rendered transparent.
slice_comparison(image, labels, titles, alpha_threshold=0.2)

interactive(children=(IntSlider(value=0, description='Slice Index:', max=18), Output()), _dom_classes=('widget…